# A deterministic Evidence Carrier research path

[View on GitHub](https://github.com/beepboop2025/liquilens-evidence-carrier/blob/main/notebooks/evidence_carrier_research.ipynb) · [Open in Colab](https://colab.research.google.com/github/beepboop2025/liquilens-evidence-carrier/blob/main/notebooks/evidence_carrier_research.ipynb) · [Launch on MyBinder](https://mybinder.org/v2/gh/beepboop2025/liquilens-evidence-carrier/main?urlpath=lab/tree/notebooks/evidence_carrier_research.ipynb)

This notebook demonstrates portable financial-evidence mechanics without a product account, API key, cookie, proprietary dataset, or user data. The values below are deliberately synthetic and make no market claim. The only runtime network operation is this notebook's first code cell: a bounded download of the immutable LiquiLens Evidence Carrier 0.14.0 wheel from GitHub Releases, verified before installation by byte count and SHA-256. Every research cell after that is local.

The notebook is public research infrastructure—not investment advice, a recommendation, an execution quote, a credit rating, or a guarantee.

In [ ]:
import hashlib
import importlib
import json
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path
from urllib.request import Request, urlopen

RELEASE = {
    "version": "0.14.0",
    "tag_commit": "8683351bd72c2a4b46d6913cd5e75c5536a410f1",
    "wheel": {
        "name": "liquilens_evidence-0.14.0-py3-none-any.whl",
        "url": "https://github.com/beepboop2025/liquilens-evidence-carrier/releases/download/v0.14.0/liquilens_evidence-0.14.0-py3-none-any.whl",
        "bytes": 43475,
        "sha256": "f0162affab57307c8e20acf91dcefc33840f91e8cf9969a8d5ec8d8df860cd24",
    },
    "sdist": {
        "url": "https://github.com/beepboop2025/liquilens-evidence-carrier/releases/download/v0.14.0/liquilens_evidence-0.14.0.tar.gz",
        "bytes": 44740,
        "sha256": "bd7a0a61bdb99784071021f95c160b9baeb22e00054f80abc03445a6cf576567",
    },
}

wheel = RELEASE["wheel"]
request = Request(wheel["url"], headers={"User-Agent": "liquilens-evidence-notebook/0.14.0"})
with urlopen(request, timeout=60) as response:
    wheel_bytes = response.read(wheel["bytes"] + 1)
assert len(wheel_bytes) == wheel["bytes"], "release wheel byte count changed"
assert hashlib.sha256(wheel_bytes).hexdigest() == wheel["sha256"], "release wheel digest changed"

with tempfile.TemporaryDirectory(prefix="liquilens-evidence-wheel-") as wheel_dir:
    wheel_path = Path(wheel_dir, wheel["name"])
    wheel_path.write_bytes(wheel_bytes)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            "--no-cache-dir",
            "--no-deps",
            "--no-index",
            "--force-reinstall",
            str(wheel_path),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
importlib.invalidate_caches()
assert version("liquilens-evidence") == RELEASE["version"]
print(json.dumps({"artifact": wheel["name"], "bytes": len(wheel_bytes), "release": RELEASE["version"], "sha256": wheel["sha256"]}, indent=2, sort_keys=True))

## 1. Issue and verify a synthetic research carrier

The example separates its clocks, declares Apache-2.0 redistribution rights, cites the exact public v1 schema digest, and retains the all-false financial-authority boundary. Fixed inputs and a fixed evaluation clock make the identity reproducible.

In [ ]:
from datetime import datetime

from liquilens_evidence import issue_evidence_carrier, verify_evidence_carrier

EVALUATED_AT = datetime.fromisoformat("2026-08-24T13:02:00+00:00")
DESCRIPTOR = {
    "producer": {"name": "liquilens", "version": "0.14.0", "endpoint": "https://liquilens.in/protocol/"},
    "subject": {"kind": "synthetic_research_series", "name": "Synthetic liquidity path", "identifiers": {"example": "synthetic-liquidity-path-v1"}},
    "claim": {"kind": "synthetic_research_example", "summary": "Synthetic values for demonstrating deterministic evidence transport", "status": "structural"},
    "clocks": {"event_time": "2026-08-24T13:00:00Z", "knowledge_time": "2026-08-24T13:01:00Z", "as_of": "2026-08-24T13:02:00Z", "expires_at": "2030-01-01T00:00:00Z"},
    "sources": [{"source_id": "liquilens:evidence-carrier-schema:v1", "publisher": "LiquiLens", "title": "LiquiLens Evidence Carrier v1 JSON Schema", "url": "https://github.com/beepboop2025/liquilens-evidence-carrier/releases/download/v0.14.0/liquilens-evidence-carrier-v1.schema.json", "retrieved_at": "2026-08-24T13:01:00Z", "content_sha256": "7f8494d8470853dc88665ea32c1dccb40cc58c55b07e9267aa28c81f83c1ccd3"}],
    "rights": {"status": "licensed", "permissions": ["ingest", "derive", "display", "redistribute"], "license": "Apache-2.0", "license_url": "https://github.com/beepboop2025/liquilens-evidence-carrier/blob/v0.14.0/LICENSE", "attribution": "LiquiLens Evidence Carrier contributors", "jurisdictions": ["global"]},
    "payload": {"series": "synthetic_liquidity_index", "unit": "index_points", "periods": ["T0", "T1", "T2"], "values": [100.0, 98.5, 101.25], "synthetic": True},
    "extensions": {"notebook": {"purpose": "deterministic_research_demo"}},
}

carrier = issue_evidence_carrier(**DESCRIPTOR)
verified = verify_evidence_carrier(carrier, evaluated_at=EVALUATED_AT)
assert carrier["record_hash"] == "52d21139aff40ef0bd1d1005183c2c92efb23959d0d4708bb678dfed0768cadf"
assert verified.disposition.value == "full"
assert verified.reason_codes == ()
assert carrier["authority"] == {"financial_authority": "none", "can_execute": False, "can_recommend": False, "is_credit_rating": False}
print(json.dumps({"authority": "none", "carrier_id": carrier["carrier_id"], "disclosure": verified.disposition.value, "record_hash": carrier["record_hash"], "rights": carrier["rights"]["status"], "synthetic": carrier["payload"]["synthetic"]}, indent=2, sort_keys=True))

## 2. Prove deterministic identity and fail closed on tampering

Issuing the same descriptor again must produce the same bytes-level identity. Changing a payload value while retaining the original hash must fail verification rather than silently producing a new claim.

In [ ]:
from copy import deepcopy

from liquilens_evidence import EvidenceCarrierError

reissued = issue_evidence_carrier(**DESCRIPTOR)
assert reissued == carrier
tampered = deepcopy(carrier)
tampered["payload"]["values"][1] = 99.0
try:
    verify_evidence_carrier(tampered, evaluated_at=EVALUATED_AT)
except EvidenceCarrierError as error:
    tamper_error = str(error)
else:
    raise AssertionError("tampered payload unexpectedly verified")
assert tamper_error == "record_hash does not match the carrier payload"
print(json.dumps({"deterministic_reissue": True, "tamper_rejected": True, "error": tamper_error}, indent=2, sort_keys=True))

## 3. Let rights—not convenience—control disclosure

A second synthetic carrier omits redistribution permission. Verification preserves its identity and provenance but exports a metadata-only reference with an explicit reason code. The reference never pretends to re-hash undisclosed payload bytes.

In [ ]:
reference_descriptor = deepcopy(DESCRIPTOR)
reference_descriptor["rights"]["permissions"] = ["ingest", "derive", "display"]
reference_source = issue_evidence_carrier(**reference_descriptor)
reference_verified = verify_evidence_carrier(reference_source, evaluated_at=EVALUATED_AT)
reference = reference_verified.export_view()
assert reference_source["record_hash"] == "2d058e66060a961316dd46e9d07c7c8623cb254d5abb798db6bb9eb6ef91c536"
assert reference_verified.disposition.value == "metadata_only"
assert reference_verified.reason_codes == ("redistribution_not_permitted",)
assert reference["payload_disclosed"] is False and "payload" not in reference
print(json.dumps({"carrier_id": reference["carrier_id"], "disclosure": reference_verified.disposition.value, "payload_present": "payload" in reference, "reason_codes": reference["reason_codes"], "record_hash": reference["record_hash"]}, indent=2, sort_keys=True))

## 4. Project the verified object without changing its authority

The same verified carrier can enter desktop and lineage systems through FDC3 and OpenLineage projections. These adapters preserve the carrier and its all-false authority boundary; they do not turn structural evidence into advice or execution authority.

In [ ]:
from liquilens_evidence import to_fdc3_context, to_openlineage_facet

fdc3 = to_fdc3_context(verified)
openlineage = to_openlineage_facet(verified)
assert fdc3["evidence"]["record_hash"] == carrier["record_hash"]
assert openlineage["carrier"]["record_hash"] == carrier["record_hash"]
assert fdc3["evidence"]["authority"]["can_execute"] is False
assert openlineage["carrier"]["authority"]["can_recommend"] is False
print(json.dumps({"carrier_id": carrier["carrier_id"], "fdc3_type": fdc3["type"], "openlineage_disposition": openlineage["disposition"], "record_hash_preserved": True}, indent=2, sort_keys=True))

## Evidence boundary

- **Observed upstream facts:** none; the three index values are embedded synthetic fixtures.
- **Product-derived context:** deterministic carrier identity, policy disposition, and adapter projections.
- **Clocks:** event, knowledge, retrieval, as-of, and evaluation times are fixed and remain distinct.
- **Rights:** the first fixture permits redistribution; the second intentionally does not and therefore yields only a reference.
- **Missing inputs:** no real venue, institution, market series, forecast, causal model, or live quote is present.
- **Authority:** execution, recommendation, and credit-rating authority remain false throughout.